# Genre Tagger Exploration
This notebook loads the predictions CSV, analyzes the threshold, lists dataset genres, and plays audio previews for selected genres.

In [ ]:
import pandas as pd
import ast
from IPython.display import Audio, display
# %% Bar plot of predicted‐genre frequencies
import matplotlib.pyplot as plt
from collections import Counter
# Load predictions
df = pd.read_csv('mvsep_genre_predictions.csv')

# Convert stringified lists back to Python lists
df['predicted_genres'] = df['predicted_genres'].apply(ast.literal_eval)
df['top_3_genres'] = df['top_3_genres'].apply(ast.literal_eval)
df['scores'] = df['scores'].apply(ast.literal_eval)
df['has_prediction'] = df['predicted_genres'].apply(lambda genres: len(genres) > 0)

# Create 'max_score' column by extracting the max score from each 'scores' list
df['max_score'] = df['scores'].apply(max)

# Show basic info
df.head()

## Threshold Analysis
Let's see how many files have at least one predicted genre with the current threshold (0.5) and look at the distribution of max confidence scores.

In [ ]:
import matplotlib.pyplot as plt

# compute fraction above threshold
threshold = 0.5
frac_above = (df['max_score'] >= threshold).mean() * 100

plt.figure(figsize=(10, 6))
plt.hist(df['max_score'], bins=20)                            # your histogram
plt.axvline(threshold, linestyle='--', label=f'Threshold = {threshold}')  # threshold marker

# annotation
plt.text(
    x=threshold + 0.02,                                  # slightly to the right of the line
    y=plt.gca().get_ylim()[1] * 0.9,                     # 90% up the y‑axis
    s=f"{frac_above:.1f}% ≥ {threshold}",
    verticalalignment='center'
)

plt.title('Distribution of Max Genre Confidence Scores')
plt.xlabel('Max Confidence Score')
plt.ylabel('Number of Songs')
plt.grid(axis='y', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


## Unique Genres in Dataset
Extract all genres appearing in top-3 lists to see the variety present.

In [ ]:
# count how often each genre was predicted
genre_counts = Counter(g for genres in df['predicted_genres'] for g in genres)

labels, counts = zip(*genre_counts.items())

plt.figure(figsize=(12, 5))
plt.bar(labels, counts)
plt.xticks(rotation=90)
plt.ylabel('Number of Songs')
plt.title('Frequency of Predicted Genres (threshold = 0.5)')
plt.tight_layout()
plt.show()

## Audio Previews by Genre
Select a genre below to listen to a few examples tagged with that genre.

In [ ]:
# Path to audio files
DATASET_DIR = '../datasets/mvsep_multisong_dataset'

def preview_genre(genre, n=3):
    subset = df[df['predicted_genres'].apply(lambda g: genre in g)]
    print(f"Showing {min(n, len(subset))} examples for genre '{genre}'")
    for _, row in subset.head(n).iterrows():
        filepath = f"{DATASET_DIR}/{row['filename']}"
        print(row['filename'], row['predicted_genres'])
        display(Audio(filepath, autoplay=False))
        
# Example: preview 'pop'
preview_genre('pop')

In [ ]:
# %% Threshold sensitivity: how many songs get ≥1 tag at different cutoffs?
import numpy as np

thresholds = np.arange(0.0, 1.01, 0.1)
tagged_counts = []

for t in thresholds:
    # for each threshold, count songs where any score > t
    mask = df['scores'].apply(lambda scores: any(s > t for s in scores))
    tagged_counts.append(mask.sum())

plt.figure()
plt.plot(thresholds, tagged_counts, marker='o')
plt.xlabel('Threshold')
plt.ylabel('Songs with ≥1 Tag')
plt.title('Effect of Threshold on Number of Tagged Songs')
plt.grid(True)
plt.show()


In [ ]:
# %% Bar plot of predicted‐genre frequencies
import matplotlib.pyplot as plt
from collections import Counter

# count how often each genre was predicted
genre_counts = Counter(g for genres in df['predicted_genres'] for g in genres)

labels, counts = zip(*genre_counts.items())

plt.figure(figsize=(12, 5))
plt.bar(labels, counts)
plt.xticks(rotation=90)
plt.ylabel('Number of Songs')
plt.title('Frequency of Predicted Genres (threshold = 0.5)')
plt.tight_layout()
plt.show()


In [ ]:
# %% Distribution of “max_score” for untagged songs
# helps decide where to set the threshold
untagged = df[df['has_prediction'] == False]['max_score']

plt.figure()
untagged.hist(bins=20)
plt.xlabel('Max Confidence Score')
plt.ylabel('Number of Untagged Songs')
plt.title('Max Score Distribution for Songs with No Tags')
plt.show()


In [ ]:
def parse_list_column(cell):
    """Safely turn a string like "['a','b']" into a Python list."""
    if isinstance(cell, str):
        try:
            return ast.literal_eval(cell)
        except (ValueError, SyntaxError):
            print(cell)
            return []
    return cell if isinstance(cell, list) else []

df['predicted_genres_parsed'] = df['predicted_genres'].apply(parse_list_column)
df['top_3_genres_parsed']    = df['top_3_genres'].   apply(parse_list_column)

df.head()

In [ ]:
# Cell 3: Bar plot of all Top‑3 genres
import matplotlib.pyplot as plt

# explode the parsed top‑3 lists into one long Series
exploded = df.explode('top_3_genres_parsed')
counts   = exploded['top_3_genres_parsed'].value_counts()

plt.figure(figsize=(12,6))
counts.plot(kind='bar')
plt.title('Distribution of Top‑3 Predicted Genres')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# get the songs with highest confidence scores
def get_top_songs(df, n=5):
    """Get the top n songs with the highest max_score."""
    return df.nlargest(n, 'max_score')[['filename', 'max_score', 'predicted_genres_parsed']]

DATASET_DIR = '../datasets/mvsep_multisong_dataset'
# Visualize
top_songs = get_top_songs(df, n=5)
for _, row in top_songs.iterrows():
    filepath = f"{DATASET_DIR}/{row['filename']}"
    print(row['filename'], row['max_score'], row['predicted_genres_parsed'])
    display(Audio(filepath, autoplay=False))
    